In [1]:
import pandas as pd

In [6]:
data=pd.read_csv('my_file (1).csv')
# Check data types of each column
print(data.dtypes)
# Check missing values per column, ranked from most to least


Rank                                 int64
Peak                                object
All Time Peak                       object
Actual gross                        object
Adjusted gross (in 2022 dollars)    object
Artist                              object
Tour title                          object
Year(s)                             object
Shows                                int64
Average gross                       object
Ref.                                object
dtype: object


In [7]:
missing = data.isnull().sum().sort_values(ascending=False)
print(missing)

All Time Peak                       14
Peak                                11
Rank                                 0
Actual gross                         0
Adjusted gross (in 2022 dollars)     0
Artist                               0
Tour title                           0
Year(s)                              0
Shows                                0
Average gross                        0
Ref.                                 0
dtype: int64


In [9]:
data.shape

(20, 11)

In [10]:
print(data[['Peak', 'All Time Peak']])

     Peak All Time Peak
0       1             2
1       1          7[2]
2    1[4]          2[5]
3    2[7]         10[7]
4    2[4]           NaN
5    2[4]         10[9]
6   2[10]           NaN
7     NaN           NaN
8     NaN           NaN
9     NaN           NaN
10    NaN           NaN
11    NaN        14[17]
12    NaN           NaN
13  1[20]           NaN
14   2[c]           NaN
15    NaN           NaN
16    NaN           NaN
17    NaN           NaN
18    NaN           NaN
19    NaN           NaN


In [13]:
import pandas as pd
import numpy as np
import re

# Load data
data = pd.read_csv('my_file (1).csv', header=None)
data.columns = ['rank', 'peak', 'all_time_peak', 'actual_gross',
                'adjusted_gross', 'artist', 'tour_title', 'years',
                'shows', 'avg_gross', 'ref']

# Clean Peak / All Time Peak — strip footnotes like [4], [c]
def clean_rank(val):
    if pd.isna(val):
        return np.nan
    val = re.sub(r'\[.*?\]', '', str(val)).strip()
    return int(val) if val.isdigit() else np.nan

data['peak'] = data['peak'].apply(clean_rank)
data['all_time_peak'] = data['all_time_peak'].apply(clean_rank)

# Clean gross columns — remove $ , and footnotes
def clean_money(val):
    if pd.isna(val):
        return np.nan
    val = re.sub(r'\[.*?\]', '', str(val)).replace('$', '').replace(',', '').strip()
    return float(val) if val else np.nan

data['actual_gross'] = data['actual_gross'].apply(clean_money)
data['adjusted_gross'] = data['adjusted_gross'].apply(clean_money)
data['avg_gross'] = data['avg_gross'].apply(clean_money)

# Preserve meaning of NaN (did not chart) instead of imputing fake values
data['charted_peak'] = data['peak'].notna()
data['charted_all_time'] = data['all_time_peak'].notna()

data.head(20)

ValueError: could not convert string to float: 'Actual\xa0gross'

In [14]:
# ============================================
# Step 2: Rename columns to clean, consistent names
# (adjust these to match what Step 1 actually printed)
# ============================================
data.columns = ['rank', 'peak', 'all_time_peak', 'actual_gross',
                'adjusted_gross', 'artist', 'tour_title', 'years',
                'shows', 'avg_gross', 'ref']

# ============================================
# Step 3: Clean Peak / All Time Peak — strip footnotes like [4], [c]
# ============================================
def clean_rank(val):
    if pd.isna(val):
        return np.nan
    val = re.sub(r'\[.*?\]', '', str(val)).strip()
    return int(val) if val.isdigit() else np.nan

data['peak'] = data['peak'].apply(clean_rank)
data['all_time_peak'] = data['all_time_peak'].apply(clean_rank)

# ============================================
# Step 4: Clean gross columns — remove $, commas, non-breaking spaces,
# footnotes — and NEVER crash on unexpected text (return NaN instead)
# ============================================
def clean_money(val):
    if pd.isna(val):
        return np.nan
    val = str(val)
    val = re.sub(r'\[.*?\]', '', val)          # remove [4], [b], etc.
    val = val.replace('\xa0', ' ')               # normalize non-breaking spaces
    val = val.replace('$', '').replace(',', '').strip()
    try:
        return float(val)
    except ValueError:
        return np.nan                            # any leftover junk text -> NaN, no crash

data['actual_gross'] = data['actual_gross'].apply(clean_money)
data['adjusted_gross'] = data['adjusted_gross'].apply(clean_money)
data['avg_gross'] = data['avg_gross'].apply(clean_money)

# ============================================
# Step 5: Preserve meaning of NaN in Peak columns (did not chart)
# ============================================
data['charted_peak'] = data['peak'].notna()
data['charted_all_time'] = data['all_time_peak'].notna()

# ============================================
# Step 6: Check for any rows that failed to convert (worth inspecting)
# ============================================
print("\nRows where actual_gross could not be parsed:")
print(data[data['actual_gross'].isna()])

data.head(20)


Rows where actual_gross could not be parsed:
   rank  peak  all_time_peak  actual_gross  adjusted_gross  artist  \
0  Rank   NaN            NaN           NaN             NaN  Artist   

   tour_title    years  shows  avg_gross   ref  charted_peak  charted_all_time  
0  Tour title  Year(s)  Shows        NaN  Ref.         False             False  


,rank,peak,all_time_peak,actual_gross,adjusted_gross,artist,tour_title,years,shows,avg_gross,ref,charted_peak,charted_all_time
0,Rank,NaN,NaN,NaN,NaN,Artist,Tour title,Year(s),Shows,NaN,Ref.,False,False
1,1,NaN,NaN,780000000.0,780000000.0,Taylor Swift,The Eras Tour †,2023–2024,56,13928571.0,[1],False,False
2,2,NaN,NaN,579800000.0,579800000.0,Beyoncé,Renaissance World Tour,2023,56,10353571.0,[3],False,False
3,3,NaN,NaN,411000000.0,560622615.0,Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,4835294.0,[6],False,False
4,4,NaN,NaN,397300000.0,454751555.0,Pink,Beautiful Trauma World Tour,2018–2019,156,2546795.0,[7],False,False
5,5,NaN,NaN,345675146.0,402844849.0,Taylor Swift,Reputation Stadium Tour,2018,53,6522173.0,[8],False,False
6,6,NaN,NaN,305158363.0,388978496.0,Madonna,The MDNA Tour,2012,88,3467709.0,[9],False,False
7,7,NaN,NaN,280000000.0,381932682.0,Celine Dion,Taking Chances World Tour,2008–2009,131,2137405.0,[11],False,False
8,7,NaN,NaN,257600000.0,257600000.0,Pink,Summer Carnival †,2023–2024,41,6282927.0,[12],False,False
9,9,NaN,NaN,256084556.0,312258401.0,Beyoncé,The Formation World Tour,2016,49,5226215.0,[13],False,False


In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

# Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Decision Tree
decision_tree = DecisionTreeRegressor(random_state=42)
decision_tree.fit(X_train, y_train)

# Random Forest
random_forest = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)
random_forest.fit(X_train, y_train)

# Gradient Boosting
gradient_boost = GradientBoostingRegressor(random_state=42)
gradient_boost.fit(X_train, y_train)

# XGBoost
xgboost_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)
xgboost_model.fit(X_train, y_train)

NameError: name 'X_train' is not defined